# RFM案例分析

RFM模型是一种常见的客户分析模型，主要用于电商、零售、会员运营、用户分层。
R、F、M分别代表Recency最近一次消费时间，Frequency消费频率，Moneatry消费金额
R=今天减去最近一次消费日期，数值越小说明客户越活跃
通过三者的评分进行占比评估，原来确立用户画像，通常情况下金额占比最大

RFM模型的基本实现过程
1、设置要计算时的截止时间节点

In [ ]:
#导包
import numpy as np
import pandas as pd
import os
import time
from sqlalchemy import create_engine
#数据可视化工具，可绘制交互式图表
import pyecharts.options as opts
from pyecharts.charts import Bar3D
from pyecharts.commons.utils import JsCode

os.getcwd()

## 准备数据

In [16]:
#定义列表，记录读取文件的列表名字
sheet_names=['2015','2016','2017','2018','会员等级']
sheet_dict=pd.read_excel('./sales.xlsx',sheet_name=sheet_names)
sheet_dict

{'2015':               会员ID         订单号       提交日期    订单金额
 0      15278002468  3000304681 2015-01-01   499.0
 1      39236378972  3000305791 2015-01-01  2588.0
 2      38722039578  3000641787 2015-01-01   498.0
 3      11049640063  3000798913 2015-01-01  1572.0
 4      35038752292  3000821546 2015-01-01    10.1
 ...            ...         ...        ...     ...
 30769  39368100847  4281994827 2015-12-31   828.0
 30770       409757  4282010457 2015-12-31   199.0
 30771  38380526114  4282017675 2015-12-31   208.0
 30772     28074988  4282019440 2015-12-31    89.0
 30773  39460363230  4282025309 2015-12-31   719.0
 
 [30774 rows x 4 columns],
 '2016':               会员ID         订单号       提交日期     订单金额
 0      39288120141  4282025766 2016-01-01    76.00
 1      39293812118  4282037929 2016-01-01  7599.00
 2      27596340905  4282038740 2016-01-01   802.00
 3      15111475509  4282043819 2016-01-01    65.00
 4      38896594001  4282051044 2016-01-01    95.00
 ...            ...         ...

In [17]:
print(type(sheet_dict))     #读取多个工作表返回字典
sheet_dict['2015']          #获取其中一个表，键值对形式
# sheet_dict['2015'].info()   #查看详细信息
# sheet_dict['2015'].describe(include='all') #include=all会查看所有列的统计信息，不填只会统计数值列

<class 'dict'>


,会员ID,订单号,提交日期,订单金额
0,15278002468,3000304681,2015-01-01,499.0
1,39236378972,3000305791,2015-01-01,2588.0
2,38722039578,3000641787,2015-01-01,498.0
3,11049640063,3000798913,2015-01-01,1572.0
4,35038752292,3000821546,2015-01-01,10.1
...,...,...,...,...
30769,39368100847,4281994827,2015-12-31,828.0
30770,409757,4282010457,2015-12-31,199.0
30771,38380526114,4282017675,2015-12-31,208.0
30772,28074988,4282019440,2015-12-31,89.0


In [ ]:
#遍历所有表，查看数据
for i in sheet_names:
    print(i)
    print(sheet_dict[i].info())
    print(sheet_dict[i].describe())

## 数据预处理

In [19]:
#遍历
for i in sheet_names[:-1]:  #除了最后一张表不要
    #删除缺失值
    sheet_dict[i]=sheet_dict[i].dropna(axis=0)
    #筛选出订单金额大于1的订单
    sheet_dict[i]=sheet_dict[i][sheet_dict[i]['订单金额']>1]
    #固定时间节点，以每年最后一天为该年份评估的截止时间
    sheet_dict[i]['max_end_data']=pd.to_datetime(sheet_dict[i]['提交日期'].dt.year.astype(str)+'-12-31')
    # sheet_dict[i]['max_end_data']=sheet_dict[i]['提交日期']+pd.offsets.YearEnd(0)

In [30]:
#合并上述表格称为一个表，使用pd.concat实现垂直合并（merge无法实现），传入concat的必须是df和series数据类型
# merge=pd.DataFrame()
# for i in sheet_names[:-1]:
#     merge=pd.concat([merge,sheet_dict[i]],ignore_index=True)
#列表推导式
merge=pd.concat([sheet_dict[i] for i in sheet_names[:-1]],ignore_index=True)
merge

,会员ID,订单号,提交日期,订单金额,max_end_data
0,15278002468,3000304681,2015-01-01,499.0,2015-12-31
1,39236378972,3000305791,2015-01-01,2588.0,2015-12-31
2,38722039578,3000641787,2015-01-01,498.0,2015-12-31
3,11049640063,3000798913,2015-01-01,1572.0,2015-12-31
4,35038752292,3000821546,2015-01-01,10.1,2015-12-31
...,...,...,...,...,...
202822,39229485704,4354225182,2018-12-31,249.0,2018-12-31
202823,39229021075,4354225188,2018-12-31,89.0,2018-12-31
202824,39288976750,4354230034,2018-12-31,48.5,2018-12-31
202825,26772630,4354230163,2018-12-31,3196.0,2018-12-31


In [34]:
#新增year列，方便后续分组
merge['year']=merge['提交日期'].dt.year
type(merge.year[1])

numpy.int32

In [42]:
#新增本订单时间距离评估年份截止时间的差值
merge['diff_date']=merge['max_end_data']-merge['提交日期']
merge

,会员ID,订单号,提交日期,订单金额,max_end_data,year,diff_date
0,15278002468,3000304681,2015-01-01,499.0,2015-12-31,2015,364 days
1,39236378972,3000305791,2015-01-01,2588.0,2015-12-31,2015,364 days
2,38722039578,3000641787,2015-01-01,498.0,2015-12-31,2015,364 days
3,11049640063,3000798913,2015-01-01,1572.0,2015-12-31,2015,364 days
4,35038752292,3000821546,2015-01-01,10.1,2015-12-31,2015,364 days
...,...,...,...,...,...,...,...
202822,39229485704,4354225182,2018-12-31,249.0,2018-12-31,2018,0 days
202823,39229021075,4354225188,2018-12-31,89.0,2018-12-31,2018,0 days
202824,39288976750,4354230034,2018-12-31,48.5,2018-12-31,2018,0 days
202825,26772630,4354230163,2018-12-31,3196.0,2018-12-31,2018,0 days


In [43]:
#将差值转换为int类型
merge['diff_date']=merge['diff_date'].dt.days
merge

,会员ID,订单号,提交日期,订单金额,max_end_data,year,diff_date
0,15278002468,3000304681,2015-01-01,499.0,2015-12-31,2015,364
1,39236378972,3000305791,2015-01-01,2588.0,2015-12-31,2015,364
2,38722039578,3000641787,2015-01-01,498.0,2015-12-31,2015,364
3,11049640063,3000798913,2015-01-01,1572.0,2015-12-31,2015,364
4,35038752292,3000821546,2015-01-01,10.1,2015-12-31,2015,364
...,...,...,...,...,...,...,...
202822,39229485704,4354225182,2018-12-31,249.0,2018-12-31,2018,0
202823,39229021075,4354225188,2018-12-31,89.0,2018-12-31,2018,0
202824,39288976750,4354230034,2018-12-31,48.5,2018-12-31,2018,0
202825,26772630,4354230163,2018-12-31,3196.0,2018-12-31,2018,0


## 数据的统计分析

In [56]:
#基于会员ID和年份，计算R，F，M
merge_RFM=merge.groupby(['会员ID','year'],as_index=False).agg({'diff_date':'min',
                                                '订单号':'count',
                                                '订单金额':'sum'
                                                })
merge_RFM.columns=['year','会员id','R','F','M']
merge_RFM.index
merge_RFM

,year,会员id,R,F,M
0,81,2016,2,2,159.1
1,267,2015,197,2,105.0
2,278,2016,2,2,548.5
3,278,2017,67,3,7137.0
4,278,2018,36,1,49.9
...,...,...,...,...,...
148586,39545536113,2016,254,1,2399.0
148587,39545538296,2017,279,1,375.0
148588,39546132904,2015,316,1,149.7
148589,39546134364,2015,242,1,49.0


In [58]:
#查看RFM分布情况
merge_RFM.iloc[:,2:].describe().T

,count,mean,std,min,25%,50%,75%,max
R,148591.0,165.524043,101.988472,0.0,79.0,156.0,255.0,365.0
F,148591.0,1.365002,2.626953,1.0,1.0,1.0,1.0,130.0
M,148591.0,1323.741329,3753.906883,1.5,69.0,189.0,1199.0,206251.8


In [60]:
#划分区间，给出DFM的评分，R：最近一次购买时间，越小越好；F：越大越好；M：越大越好
#思路1：给出指定区间树，系统自动划分
pd.cut(merge_RFM['F'],bins=3)   #pd.cut划分的区间为左开右闭
pd.cut(merge_RFM['R'],bins=3)

0          (-0.365, 121.667]
1         (121.667, 243.333]
2          (-0.365, 121.667]
3          (-0.365, 121.667]
4          (-0.365, 121.667]
                 ...        
148586      (243.333, 365.0]
148587      (243.333, 365.0]
148588      (243.333, 365.0]
148589    (121.667, 243.333]
148590    (121.667, 243.333]
Name: R, Length: 148591, dtype: category
Categories (3, interval[float64, right]): [(-0.365, 121.667] < (121.667, 243.333] < (243.333, 365.0]]

In [62]:
#给出指定边界，系统自动划分区间数
R_bins=[-1,79,156,255,365]   #注意区间划分时为左开右闭
F_bins=[0,2,5,10,130]
M_bins=[1,69,189,1199,206252]

In [64]:
#划分区间，并给出评分
merge_RFM['R_score']=pd.cut(merge_RFM['R'],bins=R_bins,labels=[i for i in range(len(R_bins)-1,0,-1)])
merge_RFM['F_score']=pd.cut(merge_RFM['F'],bins=F_bins,labels=[i for i in range(1,len(F_bins),1)])
merge_RFM['M_score']=pd.cut(merge_RFM['M'],bins=M_bins,labels=[i for i in range(1,len(M_bins),1)])
merge_RFM

,year,会员id,R,F,M,R_score,F_score,M_score
0,81,2016,2,2,159.1,4,1,2
1,267,2015,197,2,105.0,2,1,2
2,278,2016,2,2,548.5,4,1,3
3,278,2017,67,3,7137.0,4,2,4
4,278,2018,36,1,49.9,4,1,1
...,...,...,...,...,...,...,...,...
148586,39545536113,2016,254,1,2399.0,2,1,4
148587,39545538296,2017,279,1,375.0,1,1,3
148588,39546132904,2015,316,1,149.7,1,1,2
148589,39546134364,2015,242,1,49.0,2,1,1


In [65]:
#统计所有会员的年度评分，拼接
merge_RFM['ALL']=merge_RFM['R_score'].astype(str)+merge_RFM['F_score'].astype(str)+merge_RFM['M_score'].astype(str)
merge_RFM

,year,会员id,R,F,M,R_score,F_score,M_score,ALL
0,81,2016,2,2,159.1,4,1,2,412
1,267,2015,197,2,105.0,2,1,2,212
2,278,2016,2,2,548.5,4,1,3,413
3,278,2017,67,3,7137.0,4,2,4,424
4,278,2018,36,1,49.9,4,1,1,411
...,...,...,...,...,...,...,...,...,...
148586,39545536113,2016,254,1,2399.0,2,1,4,214
148587,39545538296,2017,279,1,375.0,1,1,3,113
148588,39546132904,2015,316,1,149.7,1,1,2,112
148589,39546134364,2015,242,1,49.0,2,1,1,211
